# Model Training and Evaluation – Concepts

This notebook summarises the concepts and adds explanations and small examples. All examples use small hand-made numbers or simulated data – no data set is loaded.

**Contents**
1. What is machine learning?
2. Learning paradigms and the map of methods
3. Loss functions
4. Loss, training objective and evaluation metric
5. Train/test split and k-fold cross-validation
6. Under- and overfitting
7. The bias–variance trade-off

## Libraries and settings

In [ ]:
# Libraries
import os
import platform
import warnings
from datetime import datetime
from platform import python_version

import numpy as np
import matplotlib.pyplot as plt
from sklearn.pipeline import make_pipeline
from sklearn.preprocessing import PolynomialFeatures, StandardScaler
from sklearn.linear_model import LinearRegression
from sklearn.model_selection import KFold

# Ignore warnings
warnings.filterwarnings('ignore')

## 1. What is machine learning?

> *"A computer program is said to learn from experience E with respect to some class of tasks T and performance measure
> P, if its performance at tasks in T, as measured by P, improves with experience E."* — T. Mitchell (1997)

**In data-science terms**
- **Data:** $n$ examples (rows), each with features $x$ (e.g. living area, rooms) and – in supervised learning – a
  known answer $y$ (e.g. rent).
- **Goal:** learn a rule $\hat y = \hat f(x)$ that also gives good answers for **new** examples.

**Typical applications:** price prediction (real estate, stocks), predictive maintenance (will the machine fail?),
fraud and spam detection, recommendation (bought X → also Y), customer segmentation, perception (speech, images,
driving).

**Additional explanation – why "learning"?** Classical programming writes the rule $f$ by hand. Machine learning
*estimates* the rule from examples. The quality of the result is therefore always a question of the data (how many,
how representative, how noisy) and of the generalisation to new data – which is the topic of this notebook.

## 2. Learning paradigms and the map of methods

| paradigm | data | goal | in this course |
|---|---|---|---|
| **Supervised** | labelled pairs $(x, y)$ | learn a rule $\hat y = \hat f(x)$ that predicts the known answer | regression, classification |
| **Unsupervised** | only $x$ | find groups or patterns | clustering, PCA |
| **Reinforcement** | agent & environment | learn a strategy (actions → rewards) | not covered (games, robotics, LLM fine-tuning) |

**Map of the methods**

| | regression models | classification models | unsupervised learning |
|---|---|---|---|
| target $y$ | a number (CHF, kg, minutes) | a category (yes/no, A/B/C) | none – only features $x$ |
| question | How much? | Which class? | Which groups / patterns? |
| methods | linear regression, regression tree, random forest, boosting | logistic regression, classification tree, random forest, boosting, k-NN, Naive Bayes, SVM | k-means, hierarchical clustering, PCA |
| evaluation | RMSE, $R^2$ on test data | accuracy, precision, recall, ROC/AUC | elbow plot, stability, interpretation |
| example | rent of a flat | survived the Titanic? | customer segments |

**Regression vs. classification** – same workflow, different target:

| $y$ (label) | colour | height [cm] | weight [kg] |
|---|---|---|---|
| cat | yellow | 42 | 2.5 |
| dog | black | 85 | 55.0 |
| dog | yellow | 35 | 4.5 |
| cat | black | 40 | 2.3 |

Here $y$ is a category → classification. If $y$ were the weight, it would be regression.

**A short history:** 1763 Bayes' theorem · 1805 least squares (Legendre, Gauss) · 1936 Turing machine · 1957
perceptron (Rosenblatt) · 1967 nearest neighbours (Cover & Hart) · 1984 CART (Breiman et al.) · 1990 boosting
(Schapire) · 1995 SVM (Cortes & Vapnik) · 2001 random forests (Breiman) · 2017 Transformer (Vaswani et al.).

## 3. Loss functions: learning = minimising a loss

A **loss** measures the error of *one* prediction. **Training** chooses the model with the smallest **average loss**
over the $n$ training examples:

$$\hat f = \arg\min_f \; \frac{1}{n}\sum_{i=1}^{n} L\big(y_i, f(x_i)\big)$$

| task | loss | formula | property |
|---|---|---|---|
| regression | squared loss | $(y-\hat y)^2$ | error 2 → 4, error 10 → 100: large errors weigh heavily (outlier-sensitive) |
| regression | absolute loss | $\lvert y-\hat y\rvert$ | grows linearly, more robust |
| classification | 0-1 loss | $1$ if wrong, $0$ if correct | what we count, but not smooth |
| classification | log-loss | $\log(1+e^{-m})$ | smooth, convex (logistic regression) |
| classification | hinge loss | $\max(0, 1-m)$ | convex, kink at $m=1$ (SVM) |

For classification with labels $y \in \{-1, +1\}$ and a score $s$, the **margin** $m = y \cdot s$ is positive if the
prediction is on the correct side. Log-loss and hinge loss are convex stand-ins for the 0-1 loss; both also penalise
"barely correct" predictions ($0 < m < 1$).

**What we actually care about** is the error on **new, unseen** data (the *generalisation error*). It cannot be computed
directly – we estimate it on data that were not used for fitting. The training error is typically **optimistically
biased**, because the same data were used to fit the model.

In [ ]:
# Plot regression and classification loss functions
e = np.linspace(-3, 3, 300)  # error y - y_hat (regression)
m = np.linspace(-2, 3, 300)  # margin y * score (classification)

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].plot(e, e**2, label='squared loss $(y-\\hat y)^2$')
axes[0].plot(e, np.abs(e), label='absolute loss $|y-\\hat y|$')
axes[0].set_xlabel('error $y - \\hat y$')
axes[0].set_ylabel('loss')
axes[0].set_title('Regression losses')
axes[0].legend()

axes[1].plot(m, (m <= 0).astype(float), label='0-1 loss')
axes[1].plot(m, np.log2(1 + np.exp(-m)), label='log-loss (base 2)')
axes[1].plot(m, np.maximum(0, 1 - m), label='hinge loss')
axes[1].axvline(0, color='grey', linewidth=0.8)
axes[1].set_xlabel('margin $m = y \\cdot s$  (m > 0: correct side)')
axes[1].set_title('Classification losses')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

**A graphical example.** Every candidate model has an average loss.
Training means searching for the parameter value at the bottom of the loss curve. Below we simulate rents of flats and
compute the MSE of lines with different slopes (the intercept is chosen optimally for each slope). The minimum is the
OLS solution.

In [ ]:
# Simulate rents and plot the MSE as a function of the slope
rng = np.random.default_rng(1)
area = rng.uniform(30, 140, 40)
rent = 160 + 30 * area + rng.normal(0, 250, 40)


def mse_for_slope(slope):
    """MSE of the line with this slope and the best intercept."""
    intercept = np.mean(rent - slope * area)  # optimal intercept for this slope
    return np.mean((rent - (intercept + slope * area)) ** 2)


slopes = np.linspace(5, 55, 200)
mse = [mse_for_slope(s) for s in slopes]
best = slopes[np.argmin(mse)]

fig, axes = plt.subplots(1, 2, figsize=(13, 4))
axes[0].scatter(area, rent, s=15, color='black')
for s, label in [(15, 'line A'), (best, 'line B (OLS)'), (45, 'line C')]:
    intercept = np.mean(rent - s * area)
    x = np.array([25, 145])
    axes[0].plot(
        x,
        intercept + s * x,
        label=f'{label}: slope {s:.1f}, MSE = {mse_for_slope(s) / 1e4:.1f}e4',
    )
axes[0].set_xlabel('living area [m²]')
axes[0].set_ylabel('rent [CHF/month]')
axes[0].set_title('Three candidate models')
axes[0].legend(fontsize=8)
axes[1].plot(slopes, np.array(mse) / 1e4)
axes[1].axvline(
    best,
    color='darkred',
    linestyle='--',
    label=f'minimum = OLS solution (slope {best:.1f})',
)
axes[1].set_xlabel('slope $\\beta_1$ (intercept set optimally)')
axes[1].set_ylabel('average loss: MSE [$10^4$ CHF²]')
axes[1].set_title('Loss as a function of the parameter')
axes[1].legend()
for ax in axes:
    ax.grid(alpha=0.3)
plt.show()

## 4. Loss, training objective and evaluation metric

| | meaning | examples |
|---|---|---|
| **loss** | error of ONE prediction | $(y-\hat y)^2$, log-loss, hinge, 0-1 |
| **training objective** | what the algorithm optimises | average loss (+ penalty) or a split criterion |
| **evaluation metric** | how we judge the model on new data | RMSE, $R^2$, accuracy, F1, AUC |

| model | training objective (what is minimised) | typical evaluation metric |
|---|---|---|
| linear regression (OLS) | sum of squared errors (RSS) | RMSE, $R^2$ on test data |
| logistic regression | log-loss = negative log-likelihood | accuracy, F1, AUC |
| decision tree (CART) | greedy: largest decrease in SSE / Gini / entropy per split | RMSE · accuracy, F1, AUC |
| SVM | hinge loss + $\lVert w\rVert^2$ penalty | accuracy, AUC |
| gradient boosting | any differentiable loss (squared, log-loss, …) | depends on the task |

**Key idea:** the objective we optimise and the metric we report need not be the same. A model is fitted with a loss that
is convenient to optimise (smooth, convex), but judged with a metric that is meaningful for the application.

## 5. Train/test split and k-fold cross-validation

**Hold-out split.** The data are split once into a **training set** (e.g. 80 %: fit the model, tune hyperparameters) and
a **test set** (e.g. 20 %). The test set is used **only once at the very end** – otherwise the test error is no longer an
honest estimate.

**k-fold cross-validation** (on the training set, e.g. $k = 5$): split the training data into $k$ folds; fit on $k-1$
folds and validate on the remaining one; repeat $k$ times so that each fold is used once for validation:

$$\text{CV error} = \frac{1}{k}\sum_{j=1}^{k} \text{MSE}_j$$

**Additional explanation.** CV uses every training observation for validation exactly once and averages over $k$
estimates – it is more stable than a single validation split, especially for small data sets. It is used to choose
hyperparameters (e.g. the polynomial degree, the depth of a tree). The cost is $k$ model fits.

In [ ]:
# Visualise the folds of a 5-fold CV for 20 observations
kf = KFold(n_splits=5, shuffle=True, random_state=0)
grid = np.zeros((5, 20))
for run, (train_idx, val_idx) in enumerate(kf.split(np.arange(20))):
    grid[run, val_idx] = 1

plt.figure(figsize=(9, 2.5))
plt.imshow(grid, cmap='coolwarm', aspect='auto', vmin=-0.3, vmax=1.3)
plt.yticks(range(5), [f'run {i + 1}' for i in range(5)])
plt.xticks(range(20), range(1, 21), fontsize=7)
plt.xlabel('observation (training set)')
plt.title('5-fold cross-validation: blue = train, red = validate')
plt.show()

## 6. Under- and overfitting

- **Underfitting:** model too simple – high error on training *and* test data.
- **Good fit:** captures the signal, ignores the noise.
- **Overfitting:** learns the noise – low training error, high test error.

The example below fits polynomials of degree 1, 4 and 15 to 18 noisy points of a sine curve.

In [ ]:
# Fit polynomials of degree 1, 4 and 15 to noisy points of a sine curve
rng = np.random.default_rng(3)
x = np.sort(rng.uniform(0, 1, 18))
y = np.sin(2 * np.pi * x) + rng.normal(0, 0.25, 18)
x_test = rng.uniform(0, 1, 200)
y_test = np.sin(2 * np.pi * x_test) + rng.normal(0, 0.25, 200)
x_grid = np.linspace(0, 1, 400)


def poly_model(degree):
    """Polynomial regression model of the given degree."""
    return make_pipeline(
        PolynomialFeatures(degree, include_bias=False),
        StandardScaler(),
        LinearRegression(),
    )


fig, axes = plt.subplots(1, 3, figsize=(15, 4), sharey=True)
for ax, (d, title) in zip(
    axes, [(1, 'Underfitting'), (4, 'Good fit'), (15, 'Overfitting')]
):
    m = poly_model(d).fit(x.reshape(-1, 1), y)
    train_mse = np.mean((y - m.predict(x.reshape(-1, 1))) ** 2)
    test_mse = np.mean((y_test - m.predict(x_test.reshape(-1, 1))) ** 2)
    ax.plot(
        x_grid,
        np.sin(2 * np.pi * x_grid),
        'k--',
        linewidth=1,
        label='true function',
    )
    ax.plot(
        x_grid,
        m.predict(x_grid.reshape(-1, 1)),
        linewidth=2,
        label=f'degree {d}',
    )
    ax.scatter(x, y, color='black', s=20)
    ax.set_ylim(-2, 2)
    ax.set_title(
        f'{title} (degree {d})\n'
        f'train MSE = {train_mse:.3f}, test MSE = {test_mse:.3f}'
    )
    ax.legend(loc='lower left', fontsize=8)
plt.show()

## 7. The bias–variance trade-off

For squared-error loss, the expected test error at a point $x$ decomposes into three parts:

$$E\big[(y-\hat f(x))^2\big] = \underbrace{\big(E[\hat f(x)] - f(x)\big)^2}_{\text{Bias}^2}
+ \underbrace{E\big[(\hat f(x) - E[\hat f(x)])^2\big]}_{\text{Variance}} + \underbrace{\sigma^2}_{\text{noise}}$$

- **Bias:** error from too simple assumptions – the model is *systematically* wrong.
- **Variance:** error from instability – the model changes a lot with another training sample.
- **Noise:** the part no model can remove.
- **Goal:** minimise the **sum** – not either term alone.

Example: predicting every rent by the overall mean → high bias; a degree-15 polynomial through 18 flats → high variance.

**Additional explanation – how can we "see" bias and variance?** Bias and variance are properties of the *training
procedure*: imagine drawing many training samples of the same size, fitting the model on each of them and looking at
the spread of the fitted curves. The simulation below does exactly this for the sine example (300 training samples
of 40 points each) and estimates bias² and variance for each degree.

In [ ]:
# Simulate bias² and variance of polynomial models of increasing degree
rng = np.random.default_rng(0)
x_eval = np.linspace(0.1, 0.9, 50)
f_true = np.sin(2 * np.pi * x_eval)
degrees = list(range(1, 13))
bias2, var = [], []
for d in degrees:
    preds = []
    for _ in range(300):  # 300 different training samples of 40 points
        xs = rng.uniform(0, 1, 40)
        ys = np.sin(2 * np.pi * xs) + rng.normal(0, 0.25, 40)
        preds.append(
            poly_model(d)
            .fit(xs.reshape(-1, 1), ys)
            .predict(x_eval.reshape(-1, 1))
        )
    preds = np.array(preds)
    bias2.append(np.mean((preds.mean(axis=0) - f_true) ** 2))
    var.append(np.mean(preds.var(axis=0)))

noise = 0.25**2
plt.figure(figsize=(7, 4))
plt.plot(degrees, bias2, marker='o', label='bias²')
plt.plot(degrees, var, marker='o', label='variance')
plt.plot(
    degrees,
    np.array(bias2) + np.array(var),
    marker='o',
    linewidth=2.5,
    label='bias² + variance',
)
plt.yscale('log')
plt.xlabel('polynomial degree (model complexity)')
plt.ylabel('expected squared error (log scale)')
plt.title(
    'Bias–variance trade-off (simulation, n = 40; '
    f'noise σ² = {noise:.4f} is added to all models)'
)
plt.legend()
plt.grid(alpha=0.3)
plt.show()

**Reading the plot:** bias² decreases with the model complexity (from degree 3 on the sine shape is captured), variance
increases. Their sum – the part of the expected test error that the model can influence; the noise is the same for all
models – is smallest for a medium complexity (here around degree 3–5). How quickly the variance grows depends on the amount of data: with more training
observations, flexible models become more stable.

**Ways to reduce the variance:** more data, simpler models, regularisation / pruning, averaging many models (bagging,
random forest). **Ways to reduce the bias:** more flexible models, better features.

**Key messages**
- The training error is optimistically biased – estimate the generalisation error on unseen data.
- Choose the complexity with cross-validation; use the test set only once.
- Under- and overfitting are two sides of the bias–variance trade-off.

### Jupyter notebook --footer info-- (please always provide this at the end of each notebook)

In [ ]:
# Show operating system, date/time and Python version
print('-----------------------------------')
print(os.name.upper())
print(platform.system(), '|', platform.release())
print('Datetime:', datetime.now().strftime("%Y-%m-%d %H:%M:%S"))
print('Python Version:', python_version())
print('-----------------------------------')